# Lesson 5B: Advanced RAG Examples

In this lesson, you'll explore advanced RAG patterns with practical working examples.

## Topics Covered
1. Multi-document RAG with filtering
2. Conversational RAG with memory
3. RAG with confidence scoring

## Learning Objectives
- Build RAG systems that handle multiple document types
- Implement conversational RAG with context
- Add confidence scoring to RAG responses
- Handle edge cases and improve reliability

In [ ]:
# Install required packages if not already installed
#%pip install python-dotenv
#%pip install openai
#%pip install numpy

# Load environment variables from .env file
import os
from dotenv import load_dotenv
load_dotenv()
import openai
import numpy as np
import json
from typing import List, Dict, Optional
from datetime import datetime
print("OpenAI package version:", openai.__version__)
print("NumPy version:", np.__version__)

In [ ]:
# Set up the OpenAI client with environment variables
chat_client = openai.OpenAI(
    api_key=os.getenv("OPENAI_API_KEY"),
    timeout=int(os.getenv("OPENAI_TIMEOUT", 30)),
    max_retries=int(os.getenv("MAX_RETRIES", 3)),
    base_url=os.getenv("OPENAI_ENDPOINT")    
)

print("Client configured successfully!")

In [ ]:
# Helper functions from Lesson 5
def get_embedding(text, model="text-embedding-3-small"):
    """Get embedding for a text string"""
    text = text.replace("\n", " ")
    response = chat_client.embeddings.create(
        input=[text],
        model=model
    )
    return response.data[0].embedding

def cosine_similarity(vec1, vec2):
    """Calculate cosine similarity between two vectors"""
    vec1 = np.array(vec1)
    vec2 = np.array(vec2)
    dot_product = np.dot(vec1, vec2)
    norm1 = np.linalg.norm(vec1)
    norm2 = np.linalg.norm(vec2)
    return dot_product / (norm1 * norm2)

print("✅ Helper functions loaded")

## Example 1: Multi-Document RAG with Filtering

Build a RAG system that handles different document types and uses metadata filtering.

**Real-world use case:** Company knowledge base with different departments (HR, IT, Sales)

In [ ]:
# Example 1A: Enhanced Vector Store with metadata filtering
class FilterableVectorStore:
    def __init__(self):
        self.documents = []
        self.embeddings = []
        self.metadata = []
    
    def add_document(self, text, metadata=None):
        """Add a document with metadata"""
        embedding = get_embedding(text)
        self.documents.append(text)
        self.embeddings.append(embedding)
        self.metadata.append(metadata or {})
        return len(self.documents) - 1
    
    def search(self, query, top_k=5, filters=None, min_similarity=0.0):
        """Search with optional metadata filtering"""
        query_embedding = get_embedding(query)
        
        results = []
        for i, doc_embedding in enumerate(self.embeddings):
            # Apply metadata filters
            if filters:
                if not self._matches_filters(self.metadata[i], filters):
                    continue
            
            similarity = cosine_similarity(query_embedding, doc_embedding)
            
            # Apply similarity threshold
            if similarity >= min_similarity:
                results.append({
                    'index': i,
                    'document': self.documents[i],
                    'similarity': similarity,
                    'metadata': self.metadata[i]
                })
        
        # Sort by similarity
        results.sort(key=lambda x: x['similarity'], reverse=True)
        return results[:top_k]
    
    def _matches_filters(self, metadata, filters):
        """Check if metadata matches all filters"""
        for key, value in filters.items():
            if key not in metadata or metadata[key] != value:
                return False
        return True
    
    def get_stats(self):
        """Get store statistics"""
        departments = {}
        for meta in self.metadata:
            dept = meta.get('department', 'unknown')
            departments[dept] = departments.get(dept, 0) + 1
        
        return {
            'total_documents': len(self.documents),
            'departments': departments
        }

print("✅ Filterable Vector Store created")

In [ ]:
# Example 1B: Populate with multi-department documents
company_kb = FilterableVectorStore()

documents = [
    # HR Documents
    {
        "text": "Employees receive 15 days of paid vacation annually. Requests must be submitted 2 weeks in advance through the HR portal.",
        "metadata": {"department": "HR", "category": "benefits", "topic": "vacation"}
    },
    {
        "text": "Health insurance enrollment opens during the first week of November. Plans include medical, dental, and vision coverage.",
        "metadata": {"department": "HR", "category": "benefits", "topic": "insurance"}
    },
    {
        "text": "Performance reviews are conducted annually in December. Employees should prepare self-assessments by November 30th.",
        "metadata": {"department": "HR", "category": "performance", "topic": "reviews"}
    },
    # IT Documents
    {
        "text": "To reset your password, visit the IT portal and click 'Forgot Password'. You'll receive a reset link via email.",
        "metadata": {"department": "IT", "category": "support", "topic": "password"}
    },
    {
        "text": "VPN access is required for remote work. Download the Cisco AnyConnect client from the IT downloads page.",
        "metadata": {"department": "IT", "category": "remote", "topic": "vpn"}
    },
    {
        "text": "New laptop requests should be submitted via IT ticket system. Processing takes 3-5 business days.",
        "metadata": {"department": "IT", "category": "equipment", "topic": "hardware"}
    },
    # Sales Documents
    {
        "text": "Sales quotas are set quarterly. Team leads review progress weekly and provide coaching as needed.",
        "metadata": {"department": "Sales", "category": "targets", "topic": "quotas"}
    },
    {
        "text": "CRM system must be updated daily. All customer interactions should be logged within 24 hours.",
        "metadata": {"department": "Sales", "category": "process", "topic": "crm"}
    },
    {
        "text": "Commission is paid monthly, calculated at 10% of closed deals. Bonuses available for exceeding quota.",
        "metadata": {"department": "Sales", "category": "compensation", "topic": "commission"}
    }
]

print("Adding documents to company knowledge base...\n")
for doc in documents:
    idx = company_kb.add_document(doc["text"], doc["metadata"])
    dept = doc["metadata"]["department"]
    topic = doc["metadata"]["topic"]
    print(f"✓ [{dept}] {topic}: {doc['text'][:50]}...")

stats = company_kb.get_stats()
print(f"\n📊 Knowledge Base Stats:")
print(f"   Total Documents: {stats['total_documents']}")
print(f"   Departments: {stats['departments']}")

In [ ]:
# Example 1C: RAG with department filtering
def filtered_rag_query(query, vector_store, department=None, top_k=3, min_similarity=0.6):
    """RAG query with optional department filtering"""
    
    print(f"\n{'='*80}")
    print(f"Query: {query}")
    if department:
        print(f"Filter: Department = {department}")
    print(f"{'='*80}\n")
    
    # Build filters
    filters = {"department": department} if department else None
    
    # Retrieve with filtering
    results = vector_store.search(
        query, 
        top_k=top_k, 
        filters=filters,
        min_similarity=min_similarity
    )
    
    if not results:
        return {
            'answer': "No relevant information found in the knowledge base.",
            'sources': [],
            'confidence': 'low'
        }
    
    print(f"📚 Retrieved {len(results)} documents:\n")
    for i, r in enumerate(results, 1):
        print(f"  {i}. [{r['similarity']:.3f}] {r['metadata']['department']} - {r['metadata']['topic']}")
        print(f"     {r['document'][:80]}...\n")
    
    # Build context
    context = "\n\n".join([
        f"[Source {i+1} - {r['metadata']['department']}]: {r['document']}"
        for i, r in enumerate(results)
    ])
    
    # Generate answer with 32k max tokens
    prompt = f"""Answer the question based on the context below. Be specific and cite sources using [Source N] notation.

Context:
{context}

Question: {query}

Answer:"""
    
    response = chat_client.chat.completions.create(
        model=os.getenv("OPENAI_MODEL", "gpt-4o-mini"),
        messages=[
            {"role": "system", "content": "You are a helpful company knowledge assistant. Provide accurate answers with source citations."},
            {"role": "user", "content": prompt}
        ],
        temperature=0.3,
        max_tokens=32000
    )
    
    answer = response.choices[0].message.content
    
    # Determine confidence based on similarity scores
    avg_similarity = np.mean([r['similarity'] for r in results])
    if avg_similarity >= 0.8:
        confidence = "high"
    elif avg_similarity >= 0.6:
        confidence = "medium"
    else:
        confidence = "low"
    
    print(f"🤖 Answer:\n{answer}\n")
    print(f"Confidence: {confidence.upper()} (avg similarity: {avg_similarity:.3f})")
    
    return {
        'answer': answer,
        'sources': results,
        'confidence': confidence,
        'avg_similarity': avg_similarity
    }

# Test queries
print("\n" + "="*80)
print("MULTI-DOCUMENT RAG WITH FILTERING")
print("="*80)

# Query 1: No filter (search all departments)
filtered_rag_query(
    "How do I request time off?",
    company_kb,
    department=None
)

# Query 2: IT department only
filtered_rag_query(
    "I need help with remote access",
    company_kb,
    department="IT"
)

# Query 3: Sales department only
filtered_rag_query(
    "How is my compensation calculated?",
    company_kb,
    department="Sales"
)

## Example 2: Conversational RAG with Memory

Build a RAG system that maintains conversation context across multiple turns.

**Real-world use case:** Customer support chatbot that remembers previous questions

In [ ]:
# Example 2A: Conversational RAG System
class ConversationalRAG:
    def __init__(self, vector_store, max_history=10):
        self.vector_store = vector_store
        self.conversation_history = []
        self.max_history = max_history
        self.system_prompt = """You are a helpful assistant with access to a knowledge base.
Use the provided context to answer questions accurately.
Remember previous questions in the conversation for context.
If you don't know the answer, say so - don't make things up."""
    
    def query(self, question, top_k=3, use_history=True):
        """Ask a question with conversation memory"""
        
        # Retrieve relevant documents
        results = self.vector_store.search(question, top_k=top_k)
        
        # Build context from documents
        context = "\n\n".join([
            f"[Document {i+1}]: {r['document']}"
            for i, r in enumerate(results)
        ])
        
        # Build messages with history
        messages = [{"role": "system", "content": self.system_prompt}]
        
        # Add conversation history if enabled
        if use_history and self.conversation_history:
            messages.extend(self.conversation_history)
        
        # Add current question with context
        user_message = f"""Context from knowledge base:
{context}

User question: {question}"""
        
        messages.append({"role": "user", "content": user_message})
        
        # Generate response with 32k max tokens
        response = chat_client.chat.completions.create(
            model=os.getenv("OPENAI_MODEL", "gpt-4o-mini"),
            messages=messages,
            temperature=0.3,
            max_tokens=16384 
        )
        
        answer = response.choices[0].message.content
        
        # Update conversation history (store simplified versions)
        self.conversation_history.append({"role": "user", "content": question})
        self.conversation_history.append({"role": "assistant", "content": answer})
        
        # Trim history if too long
        if len(self.conversation_history) > self.max_history * 2:
            self.conversation_history = self.conversation_history[-(self.max_history * 2):]
        
        return {
            'answer': answer,
            'sources': results,
            'turn': len(self.conversation_history) // 2 #TODO: Make logic clearer by directly filtering for user messages instead of relying on index arithmetic
        }
    
    def clear_history(self):
        """Clear conversation history"""
        self.conversation_history = []
    
    def get_conversation_summary(self):
        """Get summary of conversation"""
        turns = len(self.conversation_history) // 2 #TODO: Make logic clearer by directly filtering for user messages instead of relying on index arithmetic
        topics = []
        for i in range(0, len(self.conversation_history), 2):
            if i < len(self.conversation_history):
                user_msg = self.conversation_history[i]['content'][:50]
                topics.append(user_msg)
        
        return {
            'total_turns': turns,
            'topics_discussed': topics
        }

print("✅ Conversational RAG System created")

In [ ]:
# Example 2B: Create product support knowledge base
product_kb = FilterableVectorStore()

product_docs = [
    {
        "text": "The SmartHome Hub connects to WiFi via the mobile app. During setup, press the pairing button for 5 seconds until LED blinks blue.",
        "metadata": {"product": "SmartHome Hub", "category": "setup"}
    },
    {
        "text": "If the Hub LED is blinking red, it indicates a connection error. Check your WiFi password and router settings.",
        "metadata": {"product": "SmartHome Hub", "category": "troubleshooting"}
    },
    {
        "text": "SmartHome Hub supports up to 50 connected devices. Each device must be paired individually through the app.",
        "metadata": {"product": "SmartHome Hub", "category": "specs"}
    },
    {
        "text": "To reset the Hub, hold the reset button for 10 seconds. All settings and paired devices will be erased.",
        "metadata": {"product": "SmartHome Hub", "category": "troubleshooting"}
    },
    {
        "text": "The Hub includes 1-year warranty. Register your product within 30 days for extended support.",
        "metadata": {"product": "SmartHome Hub", "category": "warranty"}
    },
    {
        "text": "Firmware updates are automatic. The Hub checks for updates weekly and installs them overnight.",
        "metadata": {"product": "SmartHome Hub", "category": "maintenance"}
    }
]

print("Building product support knowledge base...\n")
for doc in product_docs:
    product_kb.add_document(doc["text"], doc["metadata"])
    print(f"✓ {doc['metadata']['category']}: {doc['text'][:60]}...")

print(f"\n📚 Added {product_kb.get_stats()['total_documents']} support articles")

In [ ]:
# Example 2C: Multi-turn conversation
conv_rag = ConversationalRAG(product_kb)

print("\n" + "="*80)
print("CONVERSATIONAL RAG - Customer Support Simulation")
print("="*80)

# Simulate a multi-turn support conversation
questions = [
    "How do I set up my SmartHome Hub?",
    "The LED is blinking red. What does that mean?",
    "How do I fix the connection error?",
    "If that doesn't work, how do I reset it?",
    "Will I lose my warranty if I reset it?"
]

for i, question in enumerate(questions, 1):
    print(f"\n{'─'*80}")
    print(f"Turn {i}/5")
    print(f"{'─'*80}")
    print(f"\n👤 Customer: {question}\n")
    
    result = conv_rag.query(question, top_k=2, use_history=True)
    
    print(f"🤖 Support Agent:\n{result['answer']}")
    print(f"\n📚 Sources used: {len(result['sources'])}")
    # print(f"   Similarities: {[f\{s['similarity']:.3f}\ for s in result['sources']]}")

# Show conversation summary
summary = conv_rag.get_conversation_summary()
print(f"\n{'='*80}")
print(f"\n📊 Conversation Summary:")
print(f"   Total turns: {summary['total_turns']}")
print(f"   Topics discussed:")
for i, topic in enumerate(summary['topics_discussed'], 1):
    print(f"     {i}. {topic}...")

## Example 3: RAG with Confidence Scoring and Fallback

Build a RAG system that evaluates answer confidence and has fallback strategies.

**Real-world use case:** Healthcare Q&A where accuracy is critical

In [ ]:
# Example 3A: Confidence-Aware RAG System
class ConfidenceRAG:
    def __init__(self, vector_store):
        self.vector_store = vector_store
        
        # Confidence thresholds
        self.HIGH_CONFIDENCE = 0.85
        self.MEDIUM_CONFIDENCE = 0.70
        self.LOW_CONFIDENCE = 0.50
    
    def query(self, question, top_k=5, require_high_confidence=False):
        """Query with confidence evaluation"""
        
        # Retrieve documents
        results = self.vector_store.search(question, top_k=top_k)
        
        if not results:
            return self._no_results_response(question)
        
        # Calculate confidence metrics
        similarities = [r['similarity'] for r in results]
        avg_similarity = np.mean(similarities)
        max_similarity = max(similarities)
        similarity_variance = np.var(similarities)
        
        # Determine overall confidence level
        confidence_level = self._evaluate_confidence(
            avg_similarity, 
            max_similarity, 
            similarity_variance
        )
        
        # Check if confidence requirement is met
        if require_high_confidence and confidence_level != "high":
            return self._low_confidence_response(question, results, confidence_level)
        
        # Build context with confidence indicators
        context_parts = []
        for i, r in enumerate(results[:3], 1):  # Use top 3
            confidence_indicator = "✓" if r['similarity'] >= self.HIGH_CONFIDENCE else "~"
            context_parts.append(
                f"[Source {i}] {confidence_indicator} (Relevance: {r['similarity']:.2f})\n{r['document']}"
            )
        
        context = "\n\n".join(context_parts)
        
        # Generate answer with confidence awareness
        prompt = f"""Answer the question based on the provided sources.

Instructions:
- Only use information from the sources
- Cite sources using [Source N]
- If sources don't fully answer the question, acknowledge limitations
- ✓ indicates high-confidence sources

Sources:
{context}

Question: {question}

Answer:"""
        
        response = chat_client.chat.completions.create(
            model=os.getenv("OPENAI_MODEL", "gpt-4o-mini"),
            messages=[
                {"role": "system", "content": "You are a careful assistant that only provides information supported by sources. Be honest about limitations."},
                {"role": "user", "content": prompt}
            ],
            temperature=0.2,  # Lower temperature for accuracy
            max_tokens=16384 
        )
        
        answer = response.choices[0].message.content
        
        return {
            'answer': answer,
            'confidence_level': confidence_level,
            'confidence_metrics': {
                'avg_similarity': float(avg_similarity),
                'max_similarity': float(max_similarity),
                'variance': float(similarity_variance)
            },
            'sources': results[:3],
            'all_similarities': similarities
        }
    
    def _evaluate_confidence(self, avg_sim, max_sim, variance):
        """Evaluate overall confidence level"""
        # High confidence: top results are very relevant and consistent
        if avg_sim >= self.HIGH_CONFIDENCE and variance < 0.01:
            return "high"
        
        # Medium confidence: decent relevance but some variance
        elif avg_sim >= self.MEDIUM_CONFIDENCE or max_sim >= self.HIGH_CONFIDENCE:
            return "medium"
        
        # Low confidence: poor relevance
        else:
            return "low"
    
    def _no_results_response(self, question):
        """Response when no documents found"""
        return {
            'answer': "I couldn't find any relevant information in my knowledge base to answer this question. Please consult with a qualified professional or refer to official documentation.",
            'confidence_level': 'none',
            'confidence_metrics': {},
            'sources': [],
            'all_similarities': []
        }
    
    def _low_confidence_response(self, question, results, confidence_level):
        """Response when confidence is too low"""
        return {
            'answer': f"I found some potentially relevant information, but my confidence is only {confidence_level}. For critical decisions, please verify with authoritative sources or consult a professional.",
            'confidence_level': confidence_level,
            'confidence_metrics': {},
            'sources': results,
            'requires_verification': True
        }

print("✅ Confidence-Aware RAG System created")

In [ ]:
# Example 3B: Create medical FAQ knowledge base
medical_kb = FilterableVectorStore()

medical_faqs = [
    {
        "text": "Common cold symptoms include runny nose, sore throat, coughing, and sneezing. Most colds last 7-10 days and resolve without medical treatment.",
        "metadata": {"category": "symptoms", "condition": "cold"}
    },
    {
        "text": "Flu symptoms are typically more severe than cold symptoms and include high fever (>100.4°F), body aches, fatigue, and sometimes vomiting.",
        "metadata": {"category": "symptoms", "condition": "flu"}
    },
    {
        "text": "Stay hydrated by drinking plenty of water, juice, or warm liquids. Rest is important for recovery from common illnesses.",
        "metadata": {"category": "treatment", "type": "home-care"}
    },
    {
        "text": "Seek medical attention if fever exceeds 103°F, symptoms worsen after 7 days, or you experience difficulty breathing or chest pain.",
        "metadata": {"category": "when-to-see-doctor", "urgency": "high"}
    },
    {
        "text": "Annual flu vaccination is recommended for everyone 6 months and older. Vaccines are most effective when given before flu season begins.",
        "metadata": {"category": "prevention", "type": "vaccination"}
    },
    {
        "text": "Wash hands frequently with soap and water for at least 20 seconds to prevent spread of infections.",
        "metadata": {"category": "prevention", "type": "hygiene"}
    },
    {
        "text": "Over-the-counter pain relievers like acetaminophen or ibuprofen can help reduce fever and body aches. Follow dosage instructions carefully.",
        "metadata": {"category": "treatment", "type": "medication"}
    }
]

print("Building medical FAQ knowledge base...\n")
for faq in medical_faqs:
    medical_kb.add_document(faq["text"], faq["metadata"])
    print(f"✓ {faq['metadata']['category']}: {faq['text'][:60]}...")

print(f"\n📚 Added {medical_kb.get_stats()['total_documents']} medical FAQs")

In [ ]:
# Example 3C: Test confidence-aware responses
confidence_rag = ConfidenceRAG(medical_kb)

print("\n" + "="*80)
print("CONFIDENCE-AWARE RAG - Medical FAQ System")
print("="*80)

test_questions = [
    {
        "question": "What are the symptoms of a cold?",
        "expect": "high",
        "description": "Direct match in knowledge base"
    },
    {
        "question": "How can I prevent getting sick?",
        "expect": "medium",
        "description": "Related information available"
    },
    {
        "question": "What should I do if I have chest pain?",
        "expect": "medium/high",
        "description": "Emergency information available"
    },
    {
        "question": "How do I treat a broken bone?",
        "expect": "low",
        "description": "No relevant information in knowledge base"
    }
]

for i, test in enumerate(test_questions, 1):
    print(f"\n{'─'*80}")
    print(f"Test {i}/4: {test['description']}")
    print(f"{'─'*80}")
    print(f"\n❓ Question: {test['question']}")
    print(f"   Expected confidence: {test['expect']}\n")
    
    result = confidence_rag.query(test['question'], top_k=5)
    
    print(f"💬 Answer:\n{result['answer']}\n")
    
    # Display confidence information
    confidence_icon = {
        'high': '🟢',
        'medium': '🟡',
        'low': '🟠',
        'none': '🔴'
    }.get(result['confidence_level'], '⚪')
    
    print(f"{confidence_icon} Confidence Level: {result['confidence_level'].upper()}")
    
    if result.get('confidence_metrics'):
        metrics = result['confidence_metrics']
        print(f"\n📊 Confidence Metrics:")
        print(f"   Average Similarity: {metrics.get('avg_similarity', 0):.3f}")
        print(f"   Max Similarity: {metrics.get('max_similarity', 0):.3f}")
        print(f"   Variance: {metrics.get('variance', 0):.4f}")
    
    if result.get('sources'):
        print(f"\n📚 Top Sources ({len(result['sources'])})")
        for j, source in enumerate(result['sources'], 1):
            confidence_marker = "✓" if source['similarity'] >= 0.85 else "~"
            print(f"   {j}. {confidence_marker} [{source['similarity']:.3f}] {source['metadata']['category']}")
    
    if result.get('requires_verification'):
        print(f"\n⚠️  VERIFICATION REQUIRED - Confidence too low for critical use")

print(f"\n{'='*80}")
print("\n✅ Confidence scoring helps identify when to trust RAG responses")
print("✅ Low confidence triggers appropriate warnings or fallbacks")

## Key Takeaways

### Example 1: Multi-Document RAG with Filtering
- ✅ Metadata filtering narrows search to relevant document types
- ✅ Different departments/categories need different handling
- ✅ Similarity thresholds prevent irrelevant results

### Example 2: Conversational RAG
- ✅ Conversation history provides context for follow-up questions
- ✅ History must be trimmed to avoid token limits
- ✅ References to "it" or "that" work with conversation memory

### Example 3: Confidence-Aware RAG
- ✅ Confidence scoring prevents low-quality answers
- ✅ Multiple metrics (avg, max, variance) give better confidence assessment
- ✅ Fallback responses handle low-confidence scenarios
- ✅ Critical applications need high confidence thresholds

## Best Practices Demonstrated

1. **Always set max_tokens** for comprehensive responses. This is based on selected model. Example: 16384 for gpt-4o-mini.
2. **Filter by metadata** before semantic search when possible
3. **Maintain conversation history** for multi-turn interactions
4. **Calculate confidence scores** to evaluate answer quality
5. **Provide fallback responses** when confidence is low
6. **Cite sources** with similarity scores for transparency

